# So sánh TabNet với các mô hình baseline (Khối 1)

Notebook này huấn luyện 3 mô hình baseline (Logistic Regression, Random Forest, HistGradientBoosting) trên **cùng dữ liệu đã tiền xử lý** (cùng train/val/test split, cùng 78 features) như TabNet, để so sánh công bằng cho Chương 4 của luận văn.

**Chạy sau khi đã có:** `X_train.csv`, `X_val.csv`, `X_test.csv`, `y_train.csv`, `y_val.csv`, `y_test.csv`, `label_encoder.pkl` trong `data_dir` (từ `pre.ipynb`).

Baseline dùng dữ liệu **gốc (chưa SMOTE)** để so sánh đúng với các số liệu TabNet gốc đã có trong Bảng 4.1 của luận văn. Nếu muốn so cả với TabNet+SMOTE, xem Phần 4 cuối notebook.

In [1]:
import time
import json
import pickle
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score
)
from sklearn.preprocessing import label_binarize

import warnings
warnings.filterwarnings('ignore')


## 1. Tải dữ liệu (giống pre.ipynb / TABNET.ipynb)

In [2]:
data_dir = "../data/processed/"

X_train = pd.read_csv(data_dir + "X_train.csv")
X_val   = pd.read_csv(data_dir + "X_val.csv")
X_test  = pd.read_csv(data_dir + "X_test.csv")

y_train = pd.read_csv(data_dir + "y_train.csv").values.ravel()
y_val   = pd.read_csv(data_dir + "y_val.csv").values.ravel()
y_test  = pd.read_csv(data_dir + "y_test.csv").values.ravel()

with open(data_dir + "label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

label_names = label_encoder.classes_

print(f"Train set: {X_train.shape}, Labels: {y_train.shape}")
print(f"Validation set: {X_val.shape}, Labels: {y_val.shape}")
print(f"Test set: {X_test.shape}, Labels: {y_test.shape}")
print(f"Số lớp: {len(label_names)}")
print(f"Số features: {X_train.shape[1]}")
print(f"\nNhãn các lớp:")
for idx, name in enumerate(label_names):
    print(f"  {idx}: {name}")


Train set: (224107, 78), Labels: (224107,)
Validation set: (47868, 78), Labels: (47868,)
Test set: (47996, 78), Labels: (47996,)
Số lớp: 8
Số features: 78

Nhãn các lớp:
  0: attackfree
  1: dosattack
  2: floodattack
  3: fuzzyattack
  4: iec104starvationattack
  5: mitmattack
  6: ntpddosattack
  7: portscanattack


## 2. Hàm đánh giá — dùng đúng bộ chỉ số như Bảng 4.1 (Accuracy, Macro P/R/F1, Weighted F1, Macro AUC)

In [3]:
def evaluate_model(name, y_true, y_pred, y_proba, label_names, train_time):
    """Đánh giá 1 mô hình theo đúng bộ chỉ số đã dùng cho TabNet (Bảng 4.1)."""
    acc = accuracy_score(y_true, y_pred)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )
    per_class_p, per_class_r, per_class_f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, labels=range(len(label_names)), zero_division=0
    )
    try:
        y_true_bin = label_binarize(y_true, classes=range(len(label_names)))
        macro_auc = roc_auc_score(y_true_bin, y_proba, multi_class='ovr', average='macro')
    except Exception:
        macro_auc = np.nan

    return {
        'model': name,
        'train_time_sec': round(train_time, 2),
        'accuracy': round(float(acc), 4),
        'macro_precision': round(float(macro_p), 4),
        'macro_recall': round(float(macro_r), 4),
        'macro_f1': round(float(macro_f1), 4),
        'weighted_f1': round(float(weighted_f1), 4),
        'macro_auc': round(float(macro_auc), 4) if not np.isnan(macro_auc) else None,
        'per_class_f1': {
            label_names[i]: round(float(per_class_f1[i]), 4)
            for i in range(len(label_names))
        }
    }


## 3b. Bổ sung LightGBM và XGBoost vào bảng so sánh baseline

Bài báo gốc (Data in Brief, 2024) dùng **13 mô hình baseline**, bao gồm cả **Light Gradient Boosting Machine (LightGBM)** và **Extreme Gradient Boosting (XGBoost)** — 2 mô hình này chưa có trong `baseline_comparison.json` vì chưa được cài đặt/train ở lần chạy trước.

Cell dưới đây sẽ:
1. Cài đặt `lightgbm` và `xgboost` nếu chưa có.
2. Train 2 mô hình này trên **cùng** `X_train`/`y_train`, đánh giá trên **cùng** `X_test`/`y_test` như các baseline khác (dùng lại hàm `evaluate_model` ở Mục 2, có `kappa` và `mcc`).
3. Đọc `baseline_comparison.json` đã có sẵn, thêm 2 dòng kết quả mới, rồi ghi đè lại (đồng thời cập nhật `baseline_comparison.csv`).

In [4]:
# Cài đặt LightGBM và XGBoost nếu môi trường chưa có
import sys
import subprocess

for pkg in ["lightgbm", "xgboost"]:
    try:
        __import__(pkg)
        print(f"✓ {pkg} đã được cài đặt.")
    except ImportError:
        print(f">>> Đang cài đặt {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--quiet"])
        print(f"✓ Đã cài xong {pkg}.")

>>> Đang cài đặt lightgbm ...
✓ Đã cài xong lightgbm.
>>> Đang cài đặt xgboost ...
✓ Đã cài xong xgboost.


In [5]:
import time
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, cohen_kappa_score, matthews_corrcoef
)

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


def evaluate_model_full(name, y_test, y_pred, y_proba, train_time):
    """Đánh giá model, trả về đúng bộ chỉ số như baseline_comparison.json hiện tại
    (accuracy, macro P/R/F1, weighted F1, macro AUC, kappa, mcc)."""
    res = {'model': name, 'train_time_sec': train_time}
    res['accuracy'] = accuracy_score(y_test, y_pred)
    res['macro_precision'] = precision_score(y_test, y_pred, average='macro', zero_division=0)
    res['macro_recall'] = recall_score(y_test, y_pred, average='macro', zero_division=0)
    res['macro_f1'] = f1_score(y_test, y_pred, average='macro', zero_division=0)
    res['weighted_f1'] = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    if y_proba is not None:
        try:
            n_classes = y_proba.shape[1]
            if n_classes == 2:
                res['macro_auc'] = roc_auc_score(y_test, y_proba[:, 1])
            else:
                res['macro_auc'] = roc_auc_score(
                    y_test, y_proba, multi_class='ovr', average='macro'
                )
        except Exception as e:
            print(f'    ⚠ Không tính được AUC cho {name}: {e}')
            res['macro_auc'] = None
    else:
        res['macro_auc'] = None

    res['kappa'] = cohen_kappa_score(y_test, y_pred)
    res['mcc'] = matthews_corrcoef(y_test, y_pred)
    return res


# --- Các model bổ sung, cùng convention random_state=42, n_jobs=-1 như baseline khác ---
extra_models = {
    'Light Gradient Boosting Machine': LGBMClassifier(
        random_state=42, n_jobs=-1, verbose=-1
    ),
    'Extreme Gradient Boosting': XGBClassifier(
        eval_metric='mlogloss', random_state=42, n_jobs=-1
    ),
}

extra_results = []
for name, clf in extra_models.items():
    print(f'\n>>> Đang huấn luyện {name} ...')
    t0 = time.perf_counter()
    clf.fit(X_train.values, y_train)
    train_time = time.perf_counter() - t0

    y_pred = clf.predict(X_test.values)
    y_proba = clf.predict_proba(X_test.values) if hasattr(clf, 'predict_proba') else None

    res = evaluate_model_full(name, y_test, y_pred, y_proba, train_time)
    extra_results.append(res)
    print(f'    Accuracy={res["accuracy"]:.4f}  Macro F1={res["macro_f1"]:.4f}  '
          f'Macro AUC={res["macro_auc"]}  Thời gian train={train_time:.1f}s')

print('\n✓ Đã huấn luyện xong LightGBM và XGBoost.')


>>> Đang huấn luyện Light Gradient Boosting Machine ...
    Accuracy=0.9499  Macro F1=0.1293  Macro AUC=0.5045909286374151  Thời gian train=15.6s

>>> Đang huấn luyện Extreme Gradient Boosting ...
    Accuracy=0.9992  Macro F1=0.9338  Macro AUC=0.9999841427560736  Thời gian train=30.3s

✓ Đã huấn luyện xong LightGBM và XGBoost.


In [6]:
import os

json_path = data_dir + 'baseline_comparison.json'
csv_path = data_dir + 'baseline_comparison.csv'

# Đọc kết quả baseline đã có (nếu tồn tại), tránh trùng lặp nếu chạy lại nhiều lần
if os.path.exists(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        existing_results = json.load(f)
else:
    existing_results = []

existing_names = {r['model'] for r in existing_results}
for r in extra_results:
    if r['model'] in existing_names:
        # Ghi đè nếu đã có (ví dụ chạy lại cell để cập nhật số liệu)
        existing_results = [x for x in existing_results if x['model'] != r['model']]
    existing_results.append(r)

with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(existing_results, f, indent=4, ensure_ascii=False)

df_all = pd.DataFrame(existing_results)
df_all.to_csv(csv_path, index=False)

print(f'✓ Đã cập nhật {json_path} — tổng cộng {len(existing_results)} mô hình:')
for r in existing_results:
    print(f"  - {r['model']}")

print(f'✓ Đã cập nhật {csv_path}')

✓ Đã cập nhật ../data/processed/baseline_comparison.json — tổng cộng 14 mô hình:
  - Dummy Classifier
  - Ridge Classifier
  - Logistic Regression
  - SVM - Linear Kernel
  - Linear Discriminant Analysis
  - Gradient Boosting Classifier
  - Naive Bayes
  - K Neighbors Classifier
  - Extra Trees Classifier
  - Decision Tree Classifier
  - Random Forest Classifier
  - Quadratic Discriminant Analysis
  - Light Gradient Boosting Machine
  - Extreme Gradient Boosting
✓ Đã cập nhật ../data/processed/baseline_comparison.csv


In [7]:
import time
import json
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import RidgeClassifier, LogisticRegression, SGDClassifier
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
)
from sklearn.ensemble import (
    GradientBoostingClassifier, ExtraTreesClassifier,
    RandomForestClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, cohen_kappa_score, matthews_corrcoef
)

# LightGBM / XGBoost optional - tự bỏ qua nếu chưa cài
try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False


# ============================================================
# HÀM ĐÁNH GIÁ MODEL
# ============================================================
def evaluate_model(name, y_test, y_pred, y_proba, label_names, train_time):
    res = {'model': name, 'train_time_sec': train_time}

    res['accuracy'] = accuracy_score(y_test, y_pred)
    res['macro_precision'] = precision_score(y_test, y_pred, average='macro', zero_division=0)
    res['macro_recall'] = recall_score(y_test, y_pred, average='macro', zero_division=0)
    res['macro_f1'] = f1_score(y_test, y_pred, average='macro', zero_division=0)
    res['weighted_f1'] = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    if y_proba is not None:
        try:
            n_classes = y_proba.shape[1]
            if n_classes == 2:
                res['macro_auc'] = roc_auc_score(y_test, y_proba[:, 1])
            else:
                res['macro_auc'] = roc_auc_score(
                    y_test, y_proba, multi_class='ovr', average='macro'
                )
        except Exception as e:
            print(f'    ⚠ Không tính được AUC cho {name}: {e}')
            res['macro_auc'] = None
    else:
        res['macro_auc'] = None

    res['kappa'] = cohen_kappa_score(y_test, y_pred)
    res['mcc'] = matthews_corrcoef(y_test, y_pred)

    return res


# ============================================================
# HÀM SO SÁNH BASELINE - SVM DÙNG SGDClassifier (hinge loss) CHO NHANH
# ============================================================
def run_baseline_comparison(X_train, y_train, X_test, y_test, label_names, data_dir):
    baselines = {
        'Dummy Classifier': DummyClassifier(strategy='most_frequent', random_state=42),
        'Ridge Classifier': RidgeClassifier(random_state=42),
        'Logistic Regression': LogisticRegression(
            max_iter=1000, multi_class='multinomial', random_state=42, n_jobs=-1
        ),
        # SVM tuyến tính xấp xỉ bằng SGD - nhanh, không cần predict_proba chính xác
        'SVM - Linear Kernel': SGDClassifier(
            loss='hinge', random_state=42, max_iter=1000, n_jobs=-1
        ),
        'Linear Discriminant Analysis': LinearDiscriminantAnalysis(),
        'Gradient Boosting Classifier': GradientBoostingClassifier(random_state=42),
        'Naive Bayes': GaussianNB(),
        'K Neighbors Classifier': KNeighborsClassifier(),
        'Extra Trees Classifier': ExtraTreesClassifier(
            n_estimators=200, random_state=42, n_jobs=-1
        ),
        'Decision Tree Classifier': DecisionTreeClassifier(random_state=42),
        'Random Forest Classifier': RandomForestClassifier(
            n_estimators=200, max_depth=None, random_state=42,
            n_jobs=-1, class_weight='balanced'
        ),
        'Quadratic Discriminant Analysis': QuadraticDiscriminantAnalysis(),
    }

    if HAS_XGB:
        baselines['Extreme Gradient Boosting'] = XGBClassifier(
            eval_metric='mlogloss', random_state=42, n_jobs=-1
        )
    if HAS_LGBM:
        baselines['Light Gradient Boosting Machine'] = LGBMClassifier(
            random_state=42, n_jobs=-1, verbose=-1
        )

    results = []
    for name, clf in baselines.items():
        print(f'\n>>> Đang huấn luyện {name} ...')
        try:
            t0 = time.perf_counter()
            clf.fit(X_train, y_train)
            train_time = time.perf_counter() - t0

            y_pred = clf.predict(X_test)
            # SGDClassifier(loss='hinge') không có predict_proba -> tự động None
            y_proba = clf.predict_proba(X_test) if hasattr(clf, 'predict_proba') else None

            res = evaluate_model(name, y_test, y_pred, y_proba, label_names, train_time)
            results.append(res)
            print(f'    Accuracy={res["accuracy"]:.4f}  Macro F1={res["macro_f1"]:.4f}  '
                  f'Macro AUC={res["macro_auc"]}  Thời gian train={train_time:.1f}s')
        except Exception as e:
            print(f'    ✗ Lỗi khi huấn luyện {name}: {e}')

    df_compare = pd.DataFrame([
        {
            'Model': r['model'],
            'Accuracy': r['accuracy'],
            'Macro Precision': r['macro_precision'],
            'Macro Recall': r['macro_recall'],
            'Macro F1': r['macro_f1'],
            'Weighted F1': r['weighted_f1'],
            'Macro AUC': r['macro_auc'],
            'Train time (s)': r['train_time_sec'],
        }
        for r in results
    ])
    print('\n=== BẢNG SO SÁNH BASELINE ===')
    print(df_compare.to_string(index=False))

    df_compare.to_csv(data_dir + 'baseline_comparison.csv', index=False)
    with open(data_dir + 'baseline_comparison.json', 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=4, ensure_ascii=False)
    print(f'\n✓ Đã lưu: baseline_comparison.csv / .json')
    return df_compare, results


df_compare, baseline_results = run_baseline_comparison(
    X_train.values, y_train, X_test.values, y_test, label_names, data_dir
)


>>> Đang huấn luyện Dummy Classifier ...
    Accuracy=0.9520  Macro F1=0.1219  Macro AUC=0.5  Thời gian train=0.1s

>>> Đang huấn luyện Ridge Classifier ...
    Accuracy=0.9693  Macro F1=0.6095  Macro AUC=None  Thời gian train=0.6s

>>> Đang huấn luyện Logistic Regression ...
    Accuracy=0.9959  Macro F1=0.8522  Macro AUC=0.9978419536887155  Thời gian train=24.2s

>>> Đang huấn luyện SVM - Linear Kernel ...
    Accuracy=0.9950  Macro F1=0.7183  Macro AUC=None  Thời gian train=3.1s

>>> Đang huấn luyện Linear Discriminant Analysis ...
    Accuracy=0.9850  Macro F1=0.7536  Macro AUC=0.9134011844564255  Thời gian train=3.8s

>>> Đang huấn luyện Gradient Boosting Classifier ...
    Accuracy=0.9995  Macro F1=0.8897  Macro AUC=0.8968497729005721  Thời gian train=1233.3s

>>> Đang huấn luyện Naive Bayes ...
    Accuracy=0.6648  Macro F1=0.7283  Macro AUC=0.969662696897608  Thời gian train=0.5s

>>> Đang huấn luyện K Neighbors Classifier ...
    Accuracy=0.9981  Macro F1=0.9520  Macro AUC=0.

## 4. Ghép với kết quả TabNet để có bảng đầy đủ (dùng cho Chương 4)

Điền số liệu TabNet đã có sẵn trong luận văn (Bảng 4.1, và bản TabNet+SMOTE nếu bạn đã chạy). Nếu bạn có file `results.json` (TabNet gốc) hoặc `results_smote.json` (TabNet+SMOTE) trong `data_dir`, cell dưới sẽ tự đọc; nếu không tìm thấy, dùng số đã ghi sẵn trong luận văn làm fallback.

In [8]:
import os
import json
import pandas as pd
import numpy as np

# --- Fallback: số liệu đã có trong luận văn (Bảng 4.1) và từ lần chạy TabNet+SMOTE ---
tabnet_fallback = {
    'model': 'TabNet (gốc)',
    'accuracy': 0.9962,
    'macro_precision': 0.8938,
    'macro_recall': 0.8905,
    'macro_f1': 0.8908,
    'weighted_f1': 0.9961,
    'macro_auc': 0.9995,
}
tabnet_smote_fallback = {
    'model': 'TabNet + SMOTE',
    'accuracy': 0.9961,
    'macro_precision': 0.8872,
    'macro_recall': 0.9577,
    'macro_f1': 0.9019,
    'weighted_f1': None,
    'macro_auc': 0.9993,
}


def load_tabnet_result(json_path, fallback):
    """Đọc kết quả TabNet từ file JSON; nếu không có hoặc lỗi -> dùng số liệu ghi sẵn."""
    if os.path.exists(json_path):
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                r = json.load(f)
            return {
                'model': fallback['model'],
                'accuracy': r.get('accuracy'),
                'macro_precision': r.get('macro_avg', {}).get('precision'),
                'macro_recall': r.get('macro_avg', {}).get('recall'),
                'macro_f1': r.get('macro_avg', {}).get('f1_score'),
                'weighted_f1': r.get('weighted_avg', {}).get('f1_score'),
                'macro_auc': r.get('macro_avg', {}).get('auc'),
            }
        except Exception as e:
            print(f'⚠ Lỗi khi đọc {json_path}: {e} — dùng số liệu ghi sẵn trong luận văn (Bảng 4.1).')
            return fallback
    print(f'⚠ Không tìm thấy {json_path}, dùng số liệu ghi sẵn trong luận văn (Bảng 4.1).')
    return fallback


tabnet_result = load_tabnet_result(data_dir + 'results.json', tabnet_fallback)
tabnet_smote_result = load_tabnet_result(data_dir + 'results_smote.json', tabnet_smote_fallback)

final_rows = [
    {
        'Model': tabnet_result['model'],
        'Accuracy': tabnet_result['accuracy'],
        'Macro Precision': tabnet_result['macro_precision'],
        'Macro Recall': tabnet_result['macro_recall'],
        'Macro F1': tabnet_result['macro_f1'],
        'Weighted F1': tabnet_result['weighted_f1'],
        'Macro AUC': tabnet_result['macro_auc'],
    },
    {
        'Model': tabnet_smote_result['model'],
        'Accuracy': tabnet_smote_result['accuracy'],
        'Macro Precision': tabnet_smote_result['macro_precision'],
        'Macro Recall': tabnet_smote_result['macro_recall'],
        'Macro F1': tabnet_smote_result['macro_f1'],
        'Weighted F1': tabnet_smote_result['weighted_f1'],
        'Macro AUC': tabnet_smote_result['macro_auc'],
    },
]

# Ghép TabNet + các baseline, bỏ hẳn cột 'Train time (s)'
df_final = pd.concat([
    pd.DataFrame(final_rows),
    df_compare.drop(columns=['Train time (s)'], errors='ignore')
], ignore_index=True)

# Sắp xếp theo Accuracy giảm dần, giống format bảng gốc trong ảnh
df_final = df_final.sort_values('Accuracy', ascending=False).reset_index(drop=True)

# Làm tròn cho dễ đọc khi in ra console (không ảnh hưởng file CSV lưu số đầy đủ)
df_display = df_final.copy()
numeric_cols = ['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1', 'Weighted F1', 'Macro AUC']
df_display[numeric_cols] = df_display[numeric_cols].round(4)

print('=== BẢNG SO SÁNH ĐẦY ĐỦ — dùng trực tiếp cho Chương 4 ===')
print(df_display.to_string(index=False))

df_final.to_csv(data_dir + 'final_model_comparison.csv', index=False)
print(f'\n✓ Đã lưu bảng đầy đủ: {data_dir}final_model_comparison.csv')

=== BẢNG SO SÁNH ĐẦY ĐỦ — dùng trực tiếp cho Chương 4 ===
                          Model  Accuracy  Macro Precision  Macro Recall  Macro F1  Weighted F1  Macro AUC
       Decision Tree Classifier    0.9999           0.9500        0.9996    0.9705       0.9999     0.9998
   Gradient Boosting Classifier    0.9995           0.8981        0.8956    0.8897       0.9995     0.8968
      Extreme Gradient Boosting    0.9992           0.9196        0.9543    0.9338       0.9992     1.0000
       Random Forest Classifier    0.9983           0.9628        0.9948    0.9773       0.9983     1.0000
         K Neighbors Classifier    0.9981           0.9406        0.9729    0.9520       0.9981     0.9995
         Extra Trees Classifier    0.9973           0.9292        0.9328    0.9310       0.9973     0.9998
                   TabNet (gốc)    0.9962           0.8938        0.8905    0.8908       0.9961     0.9995
                 TabNet + SMOTE    0.9962           0.8938        0.8905    0.8908    